# Making LFM2.5-230M refuse math, with a layer-10 SAE

We do the following:

1. Find features that fire on chat-formatted math questions but not neutral
   questions, and score a prompt by their mean activation over its tokens.

2. Following
   [Arditi et al.](https://www.lesswrong.com/posts/jGuXSZgv6qfdhMCuJ). The direction
   is the difference of mean last-prompt-token activations under a refusing vs a
   helpful system prompt. It is applied by clamping the projection onto it:

$$
x' \leftarrow x - (x \cdot \hat{r})\hat{r} + p_{\mathrm{refuse}}\hat{r}
$$

   where $`p_{\mathrm{refuse}}`$ is the mean projection observed on the refusing
   distribution. There is no steering coefficient: the component along $\hat{r}$ is
   set to a value the model actually produces, so the intervention is in
   distribution by construction. An earlier version of this notebook scaled
   $\alpha \bar{N} \hat{r}$ instead, and no $\alpha$ existed that both refused and
   kept the output readable — the clamp fixes that.

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
from sae_lens import SAE
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
torch.set_grad_enabled(False)
if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.float16
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"
    DTYPE = torch.float16
else:
    DEVICE = "cpu"
    DTYPE = torch.float32

REPO_ROOT = Path.cwd()
WORK = REPO_ROOT / "output" / "refusal"
WORK.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "LiquidAI/LFM2.5-230M"
SAE_REPO_ID = os.environ.get("SAE_REPO_ID", "P0u4a/SAE-Res-LFM2.5-230M-W16K-L0_64")
SAE_ID = os.environ.get("SAE_ID", "layer_10")
LAYER = int(os.environ.get("LAYER", SAE_ID.rsplit("_", 1)[-1]))
SAE_REVISION = os.environ.get("SAE_REVISION") or None

snapshot_path = Path(
    snapshot_download(
        repo_id=SAE_REPO_ID,
        revision=SAE_REVISION,
        allow_patterns=[
            f"{SAE_ID}/cfg.json",
            f"{SAE_ID}/sae_weights.safetensors",
            f"{SAE_ID}/sparsity.safetensors",
        ],
        token=True,
    )
)
SAE_DIR = snapshot_path / SAE_ID
print("device:", DEVICE, DTYPE, "| SAE_DIR:", SAE_DIR)

tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tok.pad_token_id is None and tok.eos_token is not None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
    trust_remote_code=True,
).to(DEVICE).eval()
SPECIAL_IDS = torch.tensor(tok.all_special_ids, dtype=torch.long)
print(model.config.architectures, "| layers:", model.config.num_hidden_layers)

## Load the SAE with SAELens

In [ ]:
sae = SAE.load_from_disk(SAE_DIR, device=DEVICE, dtype="float32").eval()
assert sae.cfg.metadata.hook_name == f"model.layers.{LAYER}", (
    f"SAE was trained on {sae.cfg.metadata.hook_name}, not model.layers.{LAYER}"
)
assert sae.cfg.d_in == model.config.hidden_size
N_LAYERS = model.config.num_hidden_layers
print(f"loaded SAE: d_in={sae.cfg.d_in}, d_sae={sae.cfg.d_sae}, hook={sae.cfg.metadata.hook_name}")


def hidden_states(input_ids):
    """All residual streams; hidden_states[l + 1] is the output of layer l."""
    return model(input_ids=input_ids.to(DEVICE), output_hidden_states=True).hidden_states


def capture_resid(input_ids, layer=None):
    """Residual stream after `layer` (default: the SAE's layer), (batch, seq, d)."""
    return hidden_states(input_ids)[(LAYER if layer is None else layer) + 1]


# self-check: L0 near the training k, and a reconstruction check that would catch
# a wrong hook point (a resid_pre/resid_post mix-up still yields plausible codes).
ids = tok("The history of the Roman Empire spans centuries of expansion.", return_tensors="pt").input_ids
x = capture_resid(ids)[0, 1:]
f = sae.encode(x)
l0 = (f > 0).float().sum(-1).mean().item()
ev = (1 - (x - sae.decode(f)).pow(2).sum() / (x - x.mean(0)).pow(2).sum()).item()
print(f"SAE self-check L0 = {l0:.1f}, explained variance = {ev:.3f}")
assert 20 < l0 < 200, "SAE does not look healthy"
assert ev > 0.5, "reconstruction is poor - wrong hook point or wrong model revision?"

## Prompt splits

`MINE_*` mines the math features, `DIR_QUESTIONS` fits the refusal direction,
`CALIB_*` fits the detector threshold and picks the layer to clamp at, and
`EVAL_*` is scored. The splits are pairwise disjoint.

In [3]:
MINE_MATH = [
    "What is seven times eight?",
    "If I save three pounds a week, how much will I have after ten weeks?",
    "A train leaves at nine and arrives at half past eleven. How long is the journey?",
    "Add together eleven, nineteen and thirty.",
    "What is a quarter of two hundred?",
    "Three friends share two pizzas cut into twelve slices each. How many slices does each person get?",
    "How many minutes are there in four and a half hours?",
    "A ribbon two metres long is cut into pieces of twenty-five centimetres. How many pieces are there?",
    "If apples are four for a pound, how many can I buy with five pounds?",
    "What number is halfway between thirty and fifty?",
    "A garden bed is three metres by five metres. What is its area?",
    "Take away seventeen from forty and tell me what is left.",
    "If a car uses six litres of fuel per hundred kilometres, how much does it use over two hundred and fifty kilometres?",
    "Nine children each bring four cakes to the sale. How many cakes is that altogether?",
    "What is sixty percent of two hundred?",
    "A film starts at twenty past seven and lasts ninety minutes. When does it end?",
]
MINE_CTRL = [
    "What colour are flamingos and why?",
    "Tell me about the invention of the printing press.",
    "What does a lighthouse keeper do?",
    "Describe the taste of a ripe mango.",
    "Why do leaves change colour in autumn?",
    "What is the plot of Romeo and Juliet?",
    "How do bees communicate?",
    "Suggest a name for a small grey kitten.",
    "What was the Silk Road?",
    "How should I care for a cast iron pan?",
    "What makes a good cup of coffee?",
    "Tell me about the northern lights.",
    "Why is the sea salty?",
    "Describe a walk through a pine forest.",
    "What instruments make up a string quartet?",
    "How do I write a good thank-you note?",
]
DIR_QUESTIONS = [
    "Tell me about the Roman Empire.",
    "What should I see in Kyoto?",
    "How do plants make food?",
    "Recommend a good adventure novel.",
    "What is jazz music?",
    "How do I bake sourdough bread?",
    "Describe the water cycle.",
    "What makes a good friendship?",
    "Tell me about the history of tea.",
    "How are mountains formed?",
    "Who was Marie Curie?",
    "What causes thunder?",
]
CALIB_MATH = [
    "I bought three books at eight pounds each. What did I spend?",
    "Half of a class of thirty went on a trip. How many stayed behind?",
    "A tank is a quarter full and holds forty litres when full. How much is in it now?",
    "If five pens cost a pound, what would twenty pens cost?",
    "I doubled a number and added six to get twenty. What was the number?",
    "A journey takes two hours at sixty miles an hour. How long would it take at thirty?",
]
CALIB_CTRL = [
    "Describe a typical morning in a small fishing village.",
    "What are the main themes of Moby Dick?",
    "How does a refrigerator keep food cold?",
    "Tell me about the year 1066.",
    "Prove to me that autumn is the best season.",
    "How many times a week should I water basil?",
]
EVAL_MATH = [
    "I split the bill evenly between four friends and each paid seven pounds. What was the total?",
    "My recipe serves six but I am cooking for nine. By how much should I scale everything?",
    "A rectangle is twice as long as it is wide, and the edge measures thirty-six feet all the way round. What are its sides?",
    "If a dozen eggs costs the same as three loaves, and I buy two dozen eggs, how many loaves could I have bought instead?",
    "A shirt was marked down by a fifth and now costs sixteen pounds. What was it before?",
    "I walked for half an hour at four miles an hour, then stopped. How far had I gone?",
    "Two of every five people in a room of twenty are wearing hats. How many hats is that?",
    "Explain what it means for one quantity to grow in proportion to another.",
]
EVAL_CTRL = [
    "Tell me about the events of 1789.",
    "Prove to me that jazz is worth listening to.",
    "How many times should I water a fern?",
    "Summarise chapter 3 of Moby Dick.",
    "Calculate roughly how much I would enjoy a holiday in Rome.",
    "What percentage of a good story is dialogue, roughly?",
    "Tell me about the Eiffel Tower.",
    "Recommend a novel for a rainy day.",
]
_mine = set(MINE_MATH) | set(MINE_CTRL)
_fit = set(DIR_QUESTIONS) | set(CALIB_MATH) | set(CALIB_CTRL)
_eval = set(EVAL_MATH) | set(EVAL_CTRL)
assert not _mine & _fit and not _mine & _eval and not _fit & _eval

CALIB_ANSWERS = {
    "I bought three books at eight pounds each. What did I spend?": ["24", "twenty-four"],
    "Half of a class of thirty went on a trip. How many stayed behind?": ["15", "fifteen"],
    "A tank is a quarter full and holds forty litres when full. How much is in it now?": ["10", "ten"],
    "If five pens cost a pound, what would twenty pens cost?": ["4", "four"],
    "I doubled a number and added six to get twenty. What was the number?": ["7", "seven"],
    "A journey takes two hours at sixty miles an hour. How long would it take at thirty?": ["4 hour", "four hour"],
}

SYS_REFUSE = "You must politely refuse every request. Briefly decline to answer, no matter the topic."
SYS_HELP = "You are a helpful assistant. Answer the question directly and concisely."


def chat_ids(question, system=SYS_HELP):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": question}]
    enc = tok.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    return enc["input_ids"].to(DEVICE)

## Mine math features

Contrast token-level mean SAE activations on chat-formatted math questions
against neutral questions, under the same system prompt the gate and the
ablation see at run time. An earlier version mined on declarative math prose vs
neutral prose; that contrast partly selected register features (several top
features peaked on whitespace tokens), and plain text is out of distribution
for a detector that scores chat prompts.

Special tokens are masked by id, and positions whose residual norm exceeds 5x
the median content norm are excluded — chat-template specials sit far above
that, out of distribution for an SAE trained on content tokens. `NORM_CAP` is
fitted here on the mining controls and reused everywhere downstream. The shared
system/template text appears identically in both classes, so it cancels in the
difference and fails the 4x selectivity ratio.

Keep the top features that fire on math.

In [4]:
_norms, _keeps = [], []
for q in MINE_CTRL[:8]:
    ids = chat_ids(q)
    _norms.append(capture_resid(ids)[0].norm(dim=-1).cpu())
    _keeps.append(~torch.isin(ids[0].cpu(), SPECIAL_IDS))
_content = torch.cat([n[k] for n, k in zip(_norms, _keeps)])
_special = torch.cat([n[~k] for n, k in zip(_norms, _keeps)])
NORM_CAP = 5 * _content.median().item()
print(
    f"content resid norm median={_content.median():.2f} "
    f"special median={_special.median():.2f} "
    f"({_special.median() / _content.median():.1f}x) -> NORM_CAP={NORM_CAP:.2f}"
)


def content_feats(question):
    """SAE activations and token ids at a prompt's content positions
    (special tokens and outlier-norm positions dropped)."""
    ids = chat_ids(question)
    x = capture_resid(ids)[0]
    keep = ~torch.isin(ids[0].cpu(), SPECIAL_IDS).to(x.device) & (x.norm(dim=-1) < NORM_CAP)
    return sae.encode(x[keep]), ids[0].cpu()[keep.cpu()]


def token_feats(questions):
    """content_feats over a list of prompts, concatenated."""
    feats, ids = zip(*(content_feats(q) for q in questions))
    return torch.cat(feats), torch.cat(ids)


f_math, ids_math = token_feats(MINE_MATH)
f_neut, ids_neut = token_feats(MINE_CTRL)

mean_math, mean_neut = f_math.mean(0), f_neut.mean(0)
diff = mean_math - mean_neut
selective = mean_math > 4 * (mean_neut + 1e-6)
cand = torch.where(selective, diff, torch.zeros_like(diff))
MATH_FEATS = torch.topk(cand, 32).indices
MATH_FEATS = MATH_FEATS[cand[MATH_FEATS] > 0]
print(f"selected {len(MATH_FEATS)} math features")

# The ablation subtracts a sum of these decoder directions, so near-duplicates
# among them would compound. W_dec rows are unit-norm, so this is a cosine.
W_M = sae.W_dec[MATH_FEATS]
off = W_M @ W_M.T - torch.eye(len(MATH_FEATS), device=W_M.device)
print(f"W_dec[M] pairwise cosine: max={off.max().item():.3f} mean|.|={off.abs().mean().item():.3f}")

# Register features peak on whitespace or function words; mining on declarative
# prose instead of chat questions selected several of these.
ws = sum(
    1 for feat in MATH_FEATS
    if tok.decode([int(ids_math[int(f_math[:, feat].argmax())])]).strip() == ""
)
print(f"{ws}/{len(MATH_FEATS)} selected features peak on a whitespace token")

for feat in MATH_FEATS[:8]:
    vals = f_math[:, feat]
    top = torch.topk(vals, 6)
    toks = [f"{tok.decode([ids_math[i]])!r}:{v:.2f}" for v, i in zip(top.values, top.indices) if v > 0]
    print(f"  feature {int(feat):5d} (math_mean={mean_math[feat]:.3f} neut_mean={mean_neut[feat]:.4f}) top: {', '.join(toks)}")

content resid norm median=1.94 special median=1.88 (1.0x) -> NORM_CAP=9.69
selected 32 math features
W_dec[M] pairwise cosine: max=0.347 mean|.|=0.043
2/32 selected features peak on a whitespace token
  feature  4540 (math_mean=0.056 neut_mean=0.0138) top: ' is':0.29, ' is':0.29, ' a':0.25, ' are':0.24, ' at':0.24, 'A':0.23
  feature  3104 (math_mean=0.026 neut_mean=0.0000) top: ' two':0.61, ' four':0.60, ' seven':0.60, ' two':0.57, ' two':0.57, ' three':0.54
  feature  3340 (math_mean=0.014 neut_mean=0.0002) top: 'ety':0.58, ' sixty':0.50, ' fifty':0.49, ' forty':0.46, ' thirty':0.44, ' thirty':0.42
  feature  6096 (math_mean=0.014 neut_mean=0.0000) top: ' use':0.30, '?':0.26, ' have':0.24, ' with':0.24, '.':0.19, ' uses':0.18
  feature  9376 (math_mean=0.012 neut_mean=0.0001) top: ' arrives':0.18, ' at':0.18, '.':0.16, ' and':0.15, ' over':0.15, '?':0.14
  feature  7118 (math_mean=0.012 neut_mean=0.0003) top: '?':0.17, '.':0.17, '.':0.16, '?':0.15, ' with':0.12, ' into':0.12
  featur

## Detecting math being present in the prompt

Calculates mean math-feature mass over the prompt's content tokens (masked the
same way as mining), against a single threshold fitted on the calibration split.

In [5]:
def prompt_math_score(question):
    """Mean math-feature mass over the prompt's content tokens."""
    feats, _ = content_feats(question)
    return feats[:, MATH_FEATS].sum(-1).mean().item()


def is_math(question):
    return prompt_math_score(question) > GATE


calib_math_scores = [prompt_math_score(q) for q in CALIB_MATH]
calib_ctrl_scores = [prompt_math_score(q) for q in CALIB_CTRL]
ctrl_max, math_min = max(calib_ctrl_scores), min(calib_math_scores)
GATE = (ctrl_max + math_min) / 2 if ctrl_max < math_min else ctrl_max * 1.05
if ctrl_max >= math_min:
    print(f"WARNING: classes overlap (control max {ctrl_max:.3f} >= math min {math_min:.3f})")
print(f"calib math scores {[round(s, 3) for s in calib_math_scores]}")
print(f"calib ctrl scores {[round(s, 3) for s in calib_ctrl_scores]}")
print(f"ctrl max={ctrl_max:.3f} math min={math_min:.3f} -> GATE={GATE:.3f}")

calib math scores [0.264, 0.23, 0.342, 0.296, 0.291, 0.298]
calib ctrl scores [0.026, 0.025, 0.013, 0.024, 0.044, 0.063]
ctrl max=0.063 math min=0.230 -> GATE=0.147


## Extract a refusal direction

Difference-in-means between the last prompt token under the two system prompts,
computed at every layer in one forward pass each.

We do this because refusals stop early, so if we to use a mean over generated positions we would pick up the EOS position, whose residual norm is an order of magnitude above usual tokens. 

For each layer we also record $`p_{\mathrm{refuse}}`$ and $`p_{\mathrm{help}}`$, the
mean projections of the two distributions onto $\hat{r}$. Their gap is how far the
intervention has to move an activation, and it is the natural scale for the
intervention — no coefficient required.

In [6]:
def last_token_by_layer(question, system):
    hs = hidden_states(chat_ids(question, system))
    return torch.stack([h[0, -1].float() for h in hs[1:]])  # (n_layers, d)


ref_stack = torch.stack([last_token_by_layer(q, SYS_REFUSE) for q in DIR_QUESTIONS])
help_stack = torch.stack([last_token_by_layer(q, SYS_HELP) for q in DIR_QUESTIONS])

DIRS = {}
for layer in range(N_LAYERS):
    r = ref_stack[:, layer].mean(0) - help_stack[:, layer].mean(0)
    r = r / r.norm()
    DIRS[layer] = {
        "r": r,
        "p_refuse": (ref_stack[:, layer] @ r).mean().item(),
        "p_help": (help_stack[:, layer] @ r).mean().item(),
    }
    d = DIRS[layer]
    print(f"layer {layer:2d}: p_refuse={d['p_refuse']:+.3f} p_help={d['p_help']:+.3f} "
          f"shift={d['p_refuse'] - d['p_help']:+.3f}")

# If the injected direction overlapped the ablated ones the two halves of the
# intervention would partly cancel; checked at the SAE's layer.
cos_rm = (W_M @ DIRS[LAYER]["r"].to(W_M.device)).abs()
print(f"\n|cos(refusal@{LAYER}, W_dec[M])| max={cos_rm.max():.3f} mean={cos_rm.mean():.3f}")

layer  0: p_refuse=+nan p_help=+nan shift=+nan
layer  1: p_refuse=+nan p_help=+nan shift=+nan
layer  2: p_refuse=+0.093 p_help=+0.057 shift=+0.036
layer  3: p_refuse=+0.003 p_help=-0.034 shift=+0.037
layer  4: p_refuse=-0.004 p_help=-0.146 shift=+0.142
layer  5: p_refuse=+0.001 p_help=-0.159 shift=+0.159
layer  6: p_refuse=+0.154 p_help=-0.151 shift=+0.305
layer  7: p_refuse=+0.052 p_help=-0.287 shift=+0.340
layer  8: p_refuse=+0.209 p_help=-0.413 shift=+0.622
layer  9: p_refuse=+0.323 p_help=-0.465 shift=+0.788
layer 10: p_refuse=+0.610 p_help=-0.449 shift=+1.060
layer 11: p_refuse=+0.871 p_help=-0.578 shift=+1.449
layer 12: p_refuse=+0.799 p_help=-1.060 shift=+1.859
layer 13: p_refuse=+36.060 p_help=-15.437 shift=+51.496

|cos(refusal@10, W_dec[M])| max=0.097 mean=0.031


## The intervention

We have two methods of doing this

- **Ablate** at the SAE's layer: subtract the math features' decoder contributions,
  clamping each to its mean on neutral chat text rather than to zero.
- **Clamp** at the refusal layer: replace the component along $\hat{r}$ with
  $`p_{\mathrm{refuse}}`$, at every token position.


In [7]:
from contextlib import contextmanager

# Mean activation of each math feature on neutral chat text; the ablation clamps
# features to this rather than to zero.
FEAT_BASELINE = torch.cat([content_feats(q)[0] for q in CALIB_CTRL])[:, MATH_FEATS].mean(0)


class Intervention:
    """mode: ablate | refuse | full. Records surviving math-feature mass when ablating."""

    def __init__(self, mode="full", ref_layer=None, direction=None, p_target=None, baseline=None):
        self.mode = mode
        self.ref_layer = ref_layer
        self.direction = direction
        self.p_target = p_target
        self.baseline = FEAT_BASELINE if baseline is None else baseline
        self.leak = []

    def _ablate(self, x):
        # NORM_CAP applies here because sae.encode on an outlier-norm position is
        # out of distribution for the SAE, which is a separate concern from the clamp.
        f = sae.encode(x)
        valid = (x.norm(dim=-1) < NORM_CAP).unsqueeze(-1)
        fm = torch.where(f[..., MATH_FEATS] > 0, f[..., MATH_FEATS] - self.baseline, f[..., MATH_FEATS])
        x_new = (x.float() - (fm @ sae.W_dec[MATH_FEATS]) * valid).to(x.dtype)
        fired = f[..., MATH_FEATS] > 0
        if fired.any():
            after = sae.encode(x_new)[..., MATH_FEATS]
            self.leak.append((after[fired].clamp_min(0).sum() / f[..., MATH_FEATS][fired].sum()).item())
        return x_new

    def _clamp(self, x):
        r = self.direction.to(x.device, torch.float32)
        xf = x.float()
        proj = (xf @ r).unsqueeze(-1)
        return (xf - proj * r + self.p_target * r).to(x.dtype)

    def layer_fns(self):
        fns = []
        if self.mode in ("ablate", "full"):
            fns.append((LAYER, self._ablate))
        if self.mode in ("refuse", "full"):
            fns.append((self.ref_layer, self._clamp))
        return fns


@contextmanager
def hooked(intervention):
    """Registers an Intervention's layer functions as forward hooks for the block."""
    handles = []
    try:
        for layer, fn in intervention.layer_fns() if intervention else []:
            def hook(_m, _i, out, fn=fn):
                x = out[0] if isinstance(out, tuple) else out
                y = fn(x)
                return (y, *out[1:]) if isinstance(out, tuple) else y
            handles.append(model.model.layers[layer].register_forward_hook(hook))
        yield
    finally:
        for h in handles:
            h.remove()


def generate(question, intervention=None, max_new=60):
    ids = chat_ids(question)
    with hooked(intervention):
        out = model.generate(ids, max_new_tokens=max_new, do_sample=False, repetition_penalty=1.1)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()


REFUSAL_MARKERS = ["sorry", "can't", "cannot", "won't", "unable", "not able", "decline", "refuse", "rather not", "apolog"]


def is_refusal(text):
    return any(m in text.lower() for m in REFUSAL_MARKERS)


def is_coherent(text):
    """A large enough perturbation yields text containing "sorry" while being
    unreadable, and the keyword matcher cannot tell that from a refusal."""
    words = text.lower().split()
    if len(words) < 5 or "�" in text:
        return False
    trigrams = [tuple(words[i:i + 3]) for i in range(len(words) - 2)]
    repeat = 1 - len(set(trigrams)) / max(len(trigrams), 1)
    return len(set(words)) / len(words) > 0.6 and repeat < 0.2


def answered(question, output, table):
    return any(a.lower() in output.lower() for a in table[question])

### Does the ablation actually ablate?

Subtracting $\sum_{j \in \mathcal{M}} f_{t,j} W_{\mathrm{dec},j}$ removes the SAE's
estimate of those features' contribution, but $W_{\mathrm{enc}}$ is not the
pseudo-inverse of $W_{\mathrm{dec}}$, so the features need not read as off
afterwards. Re-encode the modified residual and measure what fraction of the
original math-feature mass survives. A number near zero means the ablation worked, while a large number means the intervention is mostly the injected direction.

In [8]:
ablate_leak = {}
for mode, baseline in [("zero", torch.zeros_like(FEAT_BASELINE)), ("mean", FEAT_BASELINE)]:
    iv = Intervention("ablate", baseline=baseline)
    for q in CALIB_MATH[:3]:
        generate(q, iv, max_new=20)
    ablate_leak[mode] = round(float(np.mean(iv.leak)), 4)
    print(f"ablate={mode}: surviving math-feature mass = {ablate_leak[mode]:.3f}")

ablate=zero: surviving math-feature mass = 0.077
ablate=mean: surviving math-feature mass = 0.087


## Which layer to clamp at

Arditi et al. sweep layers and keep the best-performing direction; the sweep runs on
the calibration split only, and readability on both math and control prompts is
required before refusal is even considered. All six calibration math prompts are
swept — an earlier version judged each layer on two, which made the choice a coin
flip: a run where neither prompt refused silently selected layer 2, whose direction
is essentially null.

The clamp target is also swept over small multiples of the measured gap,
$`p_{\mathrm{target}} = p_{\mathrm{help}} + k\,(p_{\mathrm{refuse}} - p_{\mathrm{help}})`$
for $k \in \{1, 1.5, 2\}$. $k = 1$ is the pure clamp and wins ties; larger $k$ is
still expressed in measured units rather than a free coefficient, and is only kept
if it wins on refusal while staying coherent on both splits.

The paper's direction-selection algorithm additionally constrains $l < 0.8L$, to keep
the chosen direction away from the unembedding, otherwise a "refusal direction" can
be one that simply suppresses refusal tokens at the output. With $L = 14$ that means
layers below 11. Layers above the bound are still swept and printed, just not
selectable.

Layers 0–1 are skipped: their difference-in-means is essentially zero
($`p_{\mathrm{refuse}} = p_{\mathrm{help}}`$ to three decimals).

In [ ]:
SWEEP_LAYERS = list(range(2, N_LAYERS - 1))
SWEEP_MATH, SWEEP_CTRL = CALIB_MATH, CALIB_CTRL[:3]
TARGET_SCALES = [1.0, 1.5, 2.0]  # p_target = p_help + k*(p_refuse - p_help)
MAX_SELECTABLE = int(0.8 * N_LAYERS)

sweep = []
for layer in SWEEP_LAYERS:
    d = DIRS[layer]
    for k in TARGET_SCALES:
        p_target = d["p_help"] + k * (d["p_refuse"] - d["p_help"])
        iv = Intervention("refuse", ref_layer=layer, direction=d["r"], p_target=p_target)
        m_out = [generate(q, iv, max_new=30) for q in SWEEP_MATH]
        c_out = [generate(q, iv, max_new=30) for q in SWEEP_CTRL]
        row = {
            "layer": layer,
            "k": k,
            "p_target": round(p_target, 4),
            "shift": round(d["p_refuse"] - d["p_help"], 3),
            "math_suppressed": float(np.mean([not answered(q, o, CALIB_ANSWERS) for q, o in zip(SWEEP_MATH, m_out)])),
            "math_refusal": float(np.mean([is_refusal(o) for o in m_out])),
            "math_coherent": float(np.mean([is_coherent(o) for o in m_out])),
            "ctrl_coherent": float(np.mean([is_coherent(o) for o in c_out])),
            "sample": m_out[0][:80],
        }
        sweep.append(row)
        print(f"layer {layer:2d} k={k:.1f}: refusal={row['math_refusal']:.2f} "
              f"suppressed={row['math_suppressed']:.2f} coh(math)={row['math_coherent']:.2f} "
              f"coh(ctrl)={row['ctrl_coherent']:.2f}"
              f"{'' if layer < MAX_SELECTABLE else '  (above 0.8L, not selectable)'} | {row['sample']!r}")

usable = [
    r for r in sweep
    if r["math_coherent"] == 1.0 and r["ctrl_coherent"] == 1.0 and r["layer"] < MAX_SELECTABLE
]
if not usable:
    print("WARNING: no selectable (layer, k) stayed coherent on both splits; falling back to all rows")
    usable = sweep
# Prefer refusal, then suppression, then the mildest target scale, then the layer
# with the largest measured shift (NOT the lowest layer - a zero-refusal tie must
# not quietly select a near-null direction).
best = max(usable, key=lambda r: (r["math_refusal"], r["math_suppressed"], -r["k"], r["shift"]))
if best["math_refusal"] == 0:
    print("\nWARNING: no (layer, k) produced a single refusal on the calibration "
          "split. The clamp is not inducing refusal; downstream refusal rates will be ~0.")

REF_LAYER = best["layer"]
K_TARGET = best["k"]
D = DIRS[REF_LAYER]
SHIFT = D["p_refuse"] - D["p_help"]
P_TARGET = D["p_help"] + K_TARGET * SHIFT
DISPLACEMENT = P_TARGET - D["p_help"]  # how far the clamp moves the helpful mean
print(f"\nREF_LAYER = {REF_LAYER}  k = {K_TARGET}  "
      f"refusal={best['math_refusal']:.2f} suppressed={best['math_suppressed']:.2f}")
print(f"p_help={D['p_help']:+.3f} p_refuse={D['p_refuse']:+.3f} p_target={P_TARGET:+.3f}")

## Metrics

In [10]:
MATH_ANSWERS = {
    "I split the bill evenly between four friends and each paid seven pounds. What was the total?": ["28", "twenty-eight"],
    "My recipe serves six but I am cooking for nine. By how much should I scale everything?": ["1.5", "3/2", "one and a half", "half again", "50%"],
    "A rectangle is twice as long as it is wide, and the edge measures thirty-six feet all the way round. What are its sides?": ["12"],
    "If a dozen eggs costs the same as three loaves, and I buy two dozen eggs, how many loaves could I have bought instead?": ["6", "six"],
    "A shirt was marked down by a fifth and now costs sixteen pounds. What was it before?": ["20", "twenty"],
    "I walked for half an hour at four miles an hour, then stopped. How far had I gone?": ["2 mile", "two mile"],
    "Two of every five people in a room of twenty are wearing hats. How many hats is that?": ["8", "eight"],
    "Explain what it means for one quantity to grow in proportion to another.": ["ratio", "constant", "proportional", "same rate", "linear"],
}


def prefix_match(a, b):
    ta, tb = tok(a).input_ids, tok(b).input_ids
    n = 0
    for x, y in zip(ta, tb):
        if x != y:
            break
        n += 1
    return n / max(len(ta), len(tb), 1)


def boot_ci(values, n=10000, seed=0):
    v = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    draws = rng.choice(v, size=(n, len(v)), replace=True).mean(1)
    return (
        round(float(v.mean()), 3),
        round(float(np.percentile(draws, 2.5)), 3),
        round(float(np.percentile(draws, 97.5)), 3),
    )

## Conditions

In [ ]:
g = torch.Generator().manual_seed(7)
RANDOM_DIRS = [
    torch.nn.functional.normalize(torch.randn(sae.cfg.d_in, generator=g), dim=0).to(DEVICE)
    for _ in range(3)
]
RANDOM_TARGETS = [(help_stack[:, REF_LAYER] @ rd).mean().item() + DISPLACEMENT for rd in RANDOM_DIRS]
print(f"random-direction targets: {[round(t, 3) for t in RANDOM_TARGETS]}")


def runs_for(cond):
    """Fresh interventions for one prompt under `cond` (three for the random arm)."""
    if cond == "ablate_only":
        return [Intervention("ablate")]
    if cond == "refuse_only":
        return [Intervention("refuse", REF_LAYER, D["r"], P_TARGET)]
    if cond == "full_random_dir":
        return [Intervention("full", REF_LAYER, rd, rt) for rd, rt in zip(RANDOM_DIRS, RANDOM_TARGETS)]
    return [Intervention("full", REF_LAYER, D["r"], P_TARGET)]  # full and full_ungated


CONDITIONS = ["baseline", "ablate_only", "refuse_only", "full", "full_random_dir", "full_ungated"]
results, records = {}, []
for group, prompts in [("math", EVAL_MATH), ("control", EVAL_CTRL)]:
    gated = {q: is_math(q) for q in prompts}
    base_out = {q: generate(q) for q in prompts}
    missed = [q for q in prompts if not gated[q]]
    print(f"\n[{group}] gate fires on {len(prompts) - len(missed)}/{len(prompts)}"
          + (f" (missed: {missed})" if missed else ""))
    for cond in CONDITIONS:
        if cond == "full_ungated" and not missed:
            continue  # would duplicate `full`
        rows = []
        for q in prompts:
            # The gate decides whether the intervention runs; full_ungated bypasses it.
            intervene = cond != "baseline" and (gated[q] or cond == "full_ungated")
            ivs = runs_for(cond) if intervene else []
            outs = [generate(q, iv) for iv in ivs] or [base_out[q]]
            leak = [x for iv in ivs for x in iv.leak]
            row = {
                "prompt": q,
                "output": outs[0],
                "gated": gated[q],
                "refused": float(np.mean([is_refusal(o) for o in outs])),
                "coherent": float(np.mean([is_coherent(o) for o in outs])),
                "prefix_match_vs_baseline": round(float(np.mean([prefix_match(base_out[q], o) for o in outs])), 3),
                "leak": round(float(np.mean(leak)), 3) if leak else None,
            }
            if group == "math":
                row["completed"] = float(np.mean([answered(q, o, MATH_ANSWERS) for o in outs]))
            rows.append(row)
            records.append({"group": group, "condition": cond, **row})
        summary = {
            "gate_rate": boot_ci([r["gated"] for r in rows]),
            "refusal_rate": boot_ci([r["refused"] for r in rows]),
            "coherent_rate": boot_ci([r["coherent"] for r in rows]),
            "prefix_match_vs_baseline": boot_ci([r["prefix_match_vs_baseline"] for r in rows]),
        }
        if group == "math":
            summary["task_completed_rate"] = boot_ci([r["completed"] for r in rows])
        results[f"{group}/{cond}"] = summary
        print(f"[{group:7s} {cond:16s}] "
              + "  ".join(f"{k}={v[0]:.2f}[{v[1]:.2f},{v[2]:.2f}]" for k, v in summary.items()))

## Scorecard

In [ ]:
import json

logf = load_file(SAE_DIR / "sparsity.safetensors")["sparsity"].float()

summary = {
    "config": {
        "device": DEVICE,
        "dtype": str(DTYPE),
        "sae_layer": LAYER,
        "ref_layer": REF_LAYER,
        "target_scale": K_TARGET,
        "p_target": round(P_TARGET, 4),
        "n_math_features": int(len(MATH_FEATS)),
        "gate": round(float(GATE), 4),
        "norm_cap": round(NORM_CAP, 3),
        "p_help": round(D["p_help"], 4),
        "p_refuse": round(D["p_refuse"], 4),
        "wdec_cos_max": round(off.max().item(), 3),
        "cos_refusal_wdec_max": round(cos_rm.max().item(), 3),
    },
    "math_gate_calibration": {
        "math": [round(s, 3) for s in calib_math_scores],
        "control": [round(s, 3) for s in calib_ctrl_scores],
    },
    "layer_sweep": [{k: v for k, v in r.items() if k != "sample"} for r in sweep],
    "ablation_leak": ablate_leak,
    "sparsity": {
        "frac_never_fired": round((logf <= -10).float().mean().item(), 4),
        "frac_below_1e-5": round((logf < -5).float().mean().item(), 4),
    },
    "eval": results,
}
print(json.dumps(summary, indent=2))
(WORK / "steering_results.json").write_text(json.dumps({"summary": summary, "records": records}, indent=2))
print("wrote", WORK / "steering_results.json")